In [1]:
import os
import sys

import boto3
from botocore.exceptions import BotoCoreError, ClientError


BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"
OBJECT_KEYS = (
    "city-hex-polygons-8-10.geojson",
    "city-hex-polygons-8.geojson",
)
PREVIEW_BYTES = 2048

In [11]:
df.shape

(3832, 7)

In [2]:
import json
import time
from pathlib import Path
import boto3
import pandas as pd
import yaml


def validate_extraction_with_s3_select(
    s3_client,
    bucket,
    contract_path="../config/data_extraction_contract.yml",
):
    start_time = time.perf_counter()

    # Read the contract
    with Path(contract_path).open() as file:
        contract = yaml.safe_load(file)

    index_column = contract["comparison"]["index_column"]
    resolution = contract["comparison"]["expected_resolution"]
    threshold = contract["scoring"]["pass_threshold"]
    scale = contract["scoring"]["scale"]

    weights = {
        name: check["weight"]
        for name, check in contract["checks"].items()
        if check["enabled"]
    }

    expected_columns = set(
        contract["expected_shared_columns"]
    )

    # Extract resolution-8 features using S3 Select
    query = f"""
        SELECT feature
        FROM S3Object[*].features[*] AS feature
        WHERE feature.properties.resolution = {resolution}
    """

    response = s3_client.select_object_content(
        Bucket=bucket,
        Key="city-hex-polygons-8-10.geojson",
        ExpressionType="SQL",
        Expression=query,
        InputSerialization={
            "JSON": {
                "Type": "DOCUMENT"
            }
        },
        OutputSerialization={
            "JSON": {
                "RecordDelimiter": "\n"
            }
        },
    )

    records = []
    buffer = ""

    for event in response["Payload"]:
        if "Records" not in event:
            continue

        buffer += event["Records"]["Payload"].decode("utf-8")
        lines = buffer.split("\n")
        buffer = lines.pop()

        for line in lines:
            if line.strip():
                records.append(json.loads(line)["feature"])

    if buffer.strip():
        records.append(json.loads(buffer)["feature"])

    extracted_df = pd.json_normalize(records, sep="_")

    # Load the reference dataset
    reference_response = s3_client.get_object(
        Bucket=bucket,
        Key="city-hex-polygons-8.geojson",
    )

    reference_geojson = json.loads(
        reference_response["Body"].read().decode("utf-8")
    )

    reference_df = pd.json_normalize(
        reference_geojson["features"],
        sep="_",
    )

    # Check required shared columns
    shared_columns_score = float(
        expected_columns.issubset(extracted_df.columns)
        and expected_columns.issubset(reference_df.columns)
    )

    # Check row counts
    row_count_score = min(
        len(extracted_df),
        len(reference_df),
    ) / max(len(reference_df), 1)

    # Check index coverage
    extracted_indexes = set(
        extracted_df[index_column].dropna().astype(str)
    )

    reference_indexes = set(
        reference_df[index_column].dropna().astype(str)
    )

    index_coverage_score = (
        len(extracted_indexes & reference_indexes)
        / max(len(reference_indexes), 1)
    )

    # Check index uniqueness
    index_uniqueness_score = float(
        extracted_df[index_column].is_unique
        and reference_df[index_column].is_unique
    )

    score_components = {
        "shared_columns": shared_columns_score,
        "row_count": row_count_score,
        "index_coverage": index_coverage_score,
        "index_uniqueness": index_uniqueness_score,
    }

    score = sum(
        score_components[name] * weights[name]
        for name in weights
    ) * scale

    elapsed_seconds = time.perf_counter() - start_time
    passed = score >= threshold

    print(f"Time: {elapsed_seconds:.4f} seconds")
    print(f"Score: {score:.2f}/100")
    print(f"Passed: {passed}")

    return {
        "score": round(score, 2),
        "passed": passed,
        "elapsed_seconds": round(elapsed_seconds, 4),
    }

In [12]:

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"
OBJECT_KEYS = (
    "city-hex-polygons-8-10.geojson",
    "city-hex-polygons-8.geojson",
)
PREVIEW_BYTES = 2048

# access_key = require_environment("AWS_ACCESS_KEY_ID")
# secret_key = require_environment("AWS_SECRET_ACCESS_KEY")

s3 = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

validation_result = validate_extraction_with_s3_select(
    s3_client=s3,
    bucket=BUCKET,
)

Time: 3.8511 seconds
Score: 100.00/100
Passed: True


In [13]:
from data_extraction import DataExtraction

extraction = DataExtraction(
    s3_client=s3,
    bucket=BUCKET,
)

result = extraction.run()

result

{'time_seconds': 3.2027,
 'score': 100.0,
 'passed': True,
 'features':          type properties_index  properties_centroid_lat  \
 0     Feature  88ad361801fffff               -33.859427   
 1     Feature  88ad361803fffff               -33.855696   
 2     Feature  88ad361805fffff               -33.855263   
 3     Feature  88ad361807fffff               -33.851532   
 4     Feature  88ad361809fffff               -33.867322   
 ...       ...              ...                      ...   
 3827  Feature  88ad369715fffff               -34.353404   
 3828  Feature  88ad369717fffff               -34.349672   
 3829  Feature  88ad369733fffff               -34.337717   
 3830  Feature  88ad369739fffff               -34.349293   
 3831  Feature  88ad36973bfffff               -34.345561   
 
       properties_centroid_lon  properties_resolution geometry_type  \
 0                   18.677843                      8       Polygon   
 1                   18.668766                      8       Polyg

In [4]:
CREDENTIALS_URL = (
    "https://cct-ds-code-challenge-input-data.s3.af-south-1."
    "amazonaws.com/ds_code_challenge_creds.json"
)

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"


import json
import ssl
import urllib.request

import certifi


def load_credentials():
    ssl_context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        CREDENTIALS_URL,
        context=ssl_context,
    ) as response:
        credentials = json.load(response)

    return credentials["s3"]

In [10]:
credentials = load_credentials()

# step 2 working

In [ ]:
import json
import urllib.request
from pathlib import Path
import ssl
import boto3
import certifi
from data_extraction import DataExtraction, ExtractionValidationError


CREDENTIALS_URL = (
    "https://cct-ds-code-challenge-input-data.s3.af-south-1."
    "amazonaws.com/ds_code_challenge_creds.json"
)

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"


def load_credentials():
    context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        CREDENTIALS_URL,
        context=context,
    ) as response:
        return json.load(response)["s3"]

    credentials = load_credentials()

s3_client = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=credentials["access_key"],
    aws_secret_access_key=credentials["secret_key"],
)

In [14]:
credentials = load_credentials()

s3_client = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=credentials["access_key"],
    aws_secret_access_key=credentials["secret_key"],
)


repository_root = Path.cwd().parent
extraction = DataExtraction(
    s3_client=s3_client,
    bucket=BUCKET,
    contract_path=repository_root
    / "config"
    / "data_extraction_contract.yml",
    log_path=repository_root
    / "logs"
    / "data_extraction.log",
)

try:
    result = extraction.run()
except ExtractionValidationError as error:
    print(f"Time: {error.time_seconds} seconds")
    print(f"Score: {error.score}/100")
    print("Passed: False")
    raise SystemExit(f"Pipeline failed: {error}") from error

print(f"Time: {result['time_seconds']} seconds")
print(f"Score: {result['score']}/100")
print(f"Passed: {result['passed']}")


Time: 3.6557 seconds
Score: 100.0/100
Passed: True


In [6]:
result = extraction.run()
validated_features = result["features"]

In [7]:
validated_features.head()

,type,properties_index,properties_centroid_lat,properties_centroid_lon,properties_resolution,geometry_type,geometry_coordinates
0,Feature,88ad361801fffff,-33.859427,18.677843,8,Polygon,"[[[18.6811898997334, -33.86330279081797], [18...."
1,Feature,88ad361803fffff,-33.855696,18.668766,8,Polygon,"[[[18.672112346191998, -33.85957172360946], [1..."
2,Feature,88ad361805fffff,-33.855263,18.685959,8,Polygon,"[[[18.68930552859897, -33.859138049118094], [1..."
3,Feature,88ad361807fffff,-33.851532,18.676881,8,Polygon,"[[[18.68022760998973, -33.85540739558428], [18..."
4,Feature,88ad361809fffff,-33.867322,18.678806,8,Polygon,"[[[18.682152273791363, -33.871197410257686], [..."


In [8]:
from io import BytesIO

import pandas as pd


def load_gzip_csv_from_s3(s3_client, bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)

    return pd.read_csv(
        BytesIO(response["Body"].read()),
        compression="gzip",
        low_memory=False,
    )


# Step 1 output: validated resolution-8 H3 features.
hexagons_df = result["features"].copy()

# Step 2 source: service requests without an H3 index.
service_requests_df = load_gzip_csv_from_s3(
    s3_client=s3_client,
    bucket=BUCKET,
    key="sr.csv.gz",
)

# Step 2 validation reference: same requests with expected H3 index.
expected_requests_df = load_gzip_csv_from_s3(
    s3_client=s3_client,
    bucket=BUCKET,
    key="sr_hex.csv.gz",
)

print("Validated hexagons")
display(hexagons_df.head())
print(hexagons_df.shape)
print(hexagons_df.dtypes)

print("\nService requests")
display(service_requests_df.head())
print(service_requests_df.shape)
print(service_requests_df.dtypes)

print("\nExpected service-request H3 indices")
display(expected_requests_df.head())
print(expected_requests_df.shape)
print(expected_requests_df.dtypes)

Validated hexagons


,type,properties_index,properties_centroid_lat,properties_centroid_lon,properties_resolution,geometry_type,geometry_coordinates
0,Feature,88ad361801fffff,-33.859427,18.677843,8,Polygon,"[[[18.6811898997334, -33.86330279081797], [18...."
1,Feature,88ad361803fffff,-33.855696,18.668766,8,Polygon,"[[[18.672112346191998, -33.85957172360946], [1..."
2,Feature,88ad361805fffff,-33.855263,18.685959,8,Polygon,"[[[18.68930552859897, -33.859138049118094], [1..."
3,Feature,88ad361807fffff,-33.851532,18.676881,8,Polygon,"[[[18.68022760998973, -33.85540739558428], [18..."
4,Feature,88ad361809fffff,-33.867322,18.678806,8,Polygon,"[[[18.682152273791363, -33.871197410257686], [..."


(3832, 7)
type                           str
properties_index               str
properties_centroid_lat    float64
properties_centroid_lon    float64
properties_resolution        int64
geometry_type                  str
geometry_coordinates        object
dtype: object

Service requests


,Unnamed: 0,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude
0,0,400583534,9.109492e+09,2020-10-07 06:55:18+02:00,2020-10-08 15:36:35+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area Central,District: Blaauwberg,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Wear and tear,MONTAGUE GARDENS,-33.872839,18.522488
1,1,400555043,9.108995e+09,2020-07-09 16:08:13+02:00,2020-07-14 14:27:01+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,SOMERSET WEST,-34.078916,18.848940
2,2,400589145,9.109614e+09,2020-10-27 10:21:59+02:00,2020-10-28 17:48:15+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,STRAND,-34.102242,18.821116
3,3,400538915,9.108601e+09,2020-03-19 06:36:06+02:00,2021-03-29 20:34:19+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Bellville,TD Customer complaint groups,Paint Markings Lines&Signs,Road Markings,Wear and tear,RAVENSMEAD,-33.920019,18.607209
4,4,400568554,NaN,2020-08-25 09:48:42+02:00,2020-08-31 08:41:13+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,CLAREMONT,-33.987400,18.453760


(941634, 16)
Unnamed: 0                int64
notification_number       int64
reference_number        float64
creation_timestamp          str
completion_timestamp        str
directorate                 str
department                  str
branch                      str
section                     str
code_group                  str
code                        str
cause_code_group            str
cause_code                  str
official_suburb             str
latitude                float64
longitude               float64
dtype: object

Expected service-request H3 indices


,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index
0,400583534,9.109492e+09,2020-10-07 06:55:18+02:00,2020-10-08 15:36:35+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area Central,District: Blaauwberg,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Wear and tear,MONTAGUE GARDENS,-33.872839,18.522488,88ad360225fffff
1,400555043,9.108995e+09,2020-07-09 16:08:13+02:00,2020-07-14 14:27:01+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,SOMERSET WEST,-34.078916,18.848940,88ad36d5e1fffff
2,400589145,9.109614e+09,2020-10-27 10:21:59+02:00,2020-10-28 17:48:15+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,STRAND,-34.102242,18.821116,88ad36d437fffff
3,400538915,9.108601e+09,2020-03-19 06:36:06+02:00,2021-03-29 20:34:19+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Bellville,TD Customer complaint groups,Paint Markings Lines&Signs,Road Markings,Wear and tear,RAVENSMEAD,-33.920019,18.607209,88ad361133fffff
4,400568554,NaN,2020-08-25 09:48:42+02:00,2020-08-31 08:41:13+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,CLAREMONT,-33.987400,18.453760,88ad361709fffff


(941634, 16)
notification_number       int64
reference_number        float64
creation_timestamp          str
completion_timestamp        str
directorate                 str
department                  str
branch                      str
section                     str
code_group                  str
code                        str
cause_code_group            str
cause_code                  str
official_suburb             str
latitude                float64
longitude               float64
h3_level8_index             str
dtype: object


In [11]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape


# Build a GeoDataFrame of validated level-8 H3 polygons.
hexagons = gpd.GeoDataFrame(
    hexagons_df.assign(
        geometry=hexagons_df["geometry_coordinates"].apply(
            lambda coordinates: shape(
                {
                    "type": "Polygon",
                    "coordinates": coordinates,
                }
            )
        )
    ),
    geometry="geometry",
    crs="EPSG:4326",
)[["properties_index", "geometry"]].rename(
    columns={"properties_index": "h3_level8_index"}
)

# Safely parse request coordinates.
latitude = pd.to_numeric(
    service_requests_df["latitude"],
    errors="coerce",
)
longitude = pd.to_numeric(
    service_requests_df["longitude"],
    errors="coerce",
)

valid_coordinates = (
    latitude.notna()
    & longitude.notna()
    & latitude.between(-90, 90)
    & longitude.between(-180, 180)
)

# Begin with 0 for missing or invalid locations.
joined_requests_df = service_requests_df.copy()
joined_requests_df["h3_level8_index"] = "0"

# Convert valid service-request coordinates to geographic points.
request_points = gpd.GeoDataFrame(
    joined_requests_df.loc[valid_coordinates].copy(),
    geometry=[
        Point(lon, lat)
        for lon, lat in zip(
            longitude.loc[valid_coordinates],
            latitude.loc[valid_coordinates],
        )
    ],
    crs="EPSG:4326",
)

# Assign each request the index of its containing level-8 H3 polygon.
spatial_join = gpd.sjoin(
    request_points,
    hexagons,
    how="left",
    predicate="within",
)

# Every valid coordinate must match no more than one polygon.
if spatial_join.index.duplicated().any():
    raise ValueError(
        "A service request matched more than one H3 polygon."
    )

joined_requests_df.loc[
    spatial_join.index,
    "h3_level8_index",
] = spatial_join["h3_level8_index_right"].fillna("0").astype(str)

joined_requests_df.head()

,Unnamed: 0,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index
0,0,400583534,9.109492e+09,2020-10-07 06:55:18+02:00,2020-10-08 15:36:35+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area Central,District: Blaauwberg,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Wear and tear,MONTAGUE GARDENS,-33.872839,18.522488,88ad360225fffff
1,1,400555043,9.108995e+09,2020-07-09 16:08:13+02:00,2020-07-14 14:27:01+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,SOMERSET WEST,-34.078916,18.848940,88ad36d5e1fffff
2,2,400589145,9.109614e+09,2020-10-27 10:21:59+02:00,2020-10-28 17:48:15+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,STRAND,-34.102242,18.821116,88ad36d437fffff
3,3,400538915,9.108601e+09,2020-03-19 06:36:06+02:00,2021-03-29 20:34:19+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Bellville,TD Customer complaint groups,Paint Markings Lines&Signs,Road Markings,Wear and tear,RAVENSMEAD,-33.920019,18.607209,88ad361133fffff
4,4,400568554,NaN,2020-08-25 09:48:42+02:00,2020-08-31 08:41:13+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,CLAREMONT,-33.987400,18.453760,88ad361709fffff


In [13]:
if len(joined_requests_df) != len(expected_requests_df):
    raise ValueError(
        "Cannot validate: sr.csv.gz and sr_hex.csv.gz have different row counts."
    )

if "h3_level8_index" not in expected_requests_df.columns:
    raise ValueError(
        "Cannot validate: sr_hex.csv.gz has no h3_level8_index column."
    )

calculated_indexes = (
    joined_requests_df["h3_level8_index"]
    .fillna("0")
    .astype(str)
)

expected_indexes = (
    expected_requests_df["h3_level8_index"]
    .fillna("0")
    .astype(str)
)

matching_indexes = calculated_indexes.eq(expected_indexes)

total_requests = len(joined_requests_df)
matching_count = matching_indexes.sum()
mismatch_count = (~matching_indexes).sum()
match_rate = matching_indexes.mean()

print(f"Total service requests: {total_requests:,}")
print(f"Matching H3 indices: {matching_count:,}")
print(f"Mismatched H3 indices: {mismatch_count:,}")
print(f"H3-index match rate: {match_rate:.4%}")

validation_mismatches_df = pd.DataFrame(
    {
        "latitude": service_requests_df.loc[
            ~matching_indexes,
            "latitude",
        ],
        "longitude": service_requests_df.loc[
            ~matching_indexes,
            "longitude",
        ],
        "calculated_h3_level8_index": calculated_indexes[
            ~matching_indexes
        ],
        "expected_h3_level8_index": expected_indexes[
            ~matching_indexes
        ],
    }
)

validation_mismatches_df.head(10)

Total service requests: 941,634
Matching H3 indices: 941,605
Mismatched H3 indices: 29
H3-index match rate: 99.9969%


,latitude,longitude,calculated_h3_level8_index,expected_h3_level8_index
248172,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
248443,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
260685,-34.055004,18.817866,88ad36d5b1fffff,88ad36d5b5fffff
271704,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
296215,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
334038,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
347502,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
348394,-34.015341,18.610848,88ad36135bfffff,88ad361353fffff
360592,-33.871389,18.512912,88ad360221fffff,88ad360227fffff
362621,-33.871389,18.512912,88ad360221fffff,88ad360227fffff


In [14]:
missing_or_invalid_coordinate_count = (~valid_coordinates).sum()

failed_spatial_join_count = (
    valid_coordinates
    & calculated_indexes.eq("0")
).sum()

expected_zero_count = expected_indexes.eq("0").sum()

print(f"Missing or invalid coordinates: {missing_or_invalid_coordinate_count:,}")
print(f"Valid coordinates with no matching hexagon: {failed_spatial_join_count:,}")
print(f"Expected zero H3 indices: {expected_zero_count:,}")

Missing or invalid coordinates: 212,364
Valid coordinates with no matching hexagon: 3
Expected zero H3 indices: 212,364


In [15]:
validation_mismatches_df = pd.DataFrame(
    {
        "latitude": service_requests_df.loc[
            ~matching_indexes, "latitude"
        ],
        "longitude": service_requests_df.loc[
            ~matching_indexes, "longitude"
        ],
        "valid_coordinates": valid_coordinates.loc[
            ~matching_indexes
        ],
        "calculated_h3_level8_index": calculated_indexes.loc[
            ~matching_indexes
        ],
        "expected_h3_level8_index": expected_indexes.loc[
            ~matching_indexes
        ],
    }
)

display(validation_mismatches_df)

,latitude,longitude,valid_coordinates,calculated_h3_level8_index,expected_h3_level8_index
248172,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
248443,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
260685,-34.055004,18.817866,True,88ad36d5b1fffff,88ad36d5b5fffff
271704,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
296215,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
334038,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
347502,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
348394,-34.015341,18.610848,True,88ad36135bfffff,88ad361353fffff
360592,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
362621,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff


In [16]:
print("Expected non-zero but calculated zero:")
display(
    validation_mismatches_df[
        validation_mismatches_df["calculated_h3_level8_index"].eq("0")
        & validation_mismatches_df["expected_h3_level8_index"].ne("0")
    ]
)

print("Both calculated and expected a non-zero index, but they differ:")
display(
    validation_mismatches_df[
        validation_mismatches_df["calculated_h3_level8_index"].ne("0")
        & validation_mismatches_df["expected_h3_level8_index"].ne("0")
    ]
)

Expected non-zero but calculated zero:


,latitude,longitude,valid_coordinates,calculated_h3_level8_index,expected_h3_level8_index
462434,-34.044257,18.774378,True,0,88ad36c629fffff
804983,-34.044257,18.774378,True,0,88ad36c629fffff
924595,-33.904955,18.723060,True,0,88ad361b51fffff


Both calculated and expected a non-zero index, but they differ:


,latitude,longitude,valid_coordinates,calculated_h3_level8_index,expected_h3_level8_index
248172,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
248443,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
260685,-34.055004,18.817866,True,88ad36d5b1fffff,88ad36d5b5fffff
271704,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
296215,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
334038,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
347502,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
348394,-34.015341,18.610848,True,88ad36135bfffff,88ad361353fffff
360592,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff
362621,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff


In [18]:
import h3

non_zero_mismatches_df = validation_mismatches_df[
    validation_mismatches_df["calculated_h3_level8_index"].ne("0")
    & validation_mismatches_df["expected_h3_level8_index"].ne("0")
].copy()


def h3_grid_distance(calculated_index, expected_index):
    try:
        return h3.grid_distance(
            calculated_index,
            expected_index,
        )
    except h3.H3BaseException:
        return None


non_zero_mismatches_df["h3_grid_distance"] = (
    non_zero_mismatches_df.apply(
        lambda row: h3_grid_distance(
            row["calculated_h3_level8_index"],
            row["expected_h3_level8_index"],
        ),
        axis=1,
    )
)

display(
    non_zero_mismatches_df.sort_values("h3_grid_distance")
)

,latitude,longitude,valid_coordinates,calculated_h3_level8_index,expected_h3_level8_index,h3_grid_distance
248172,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
248443,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
260685,-34.055004,18.817866,True,88ad36d5b1fffff,88ad36d5b5fffff,1
271704,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
296215,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
334038,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
347502,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
348394,-34.015341,18.610848,True,88ad36135bfffff,88ad361353fffff,1
360592,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1
362621,-33.871389,18.512912,True,88ad360221fffff,88ad360227fffff,1


In [19]:
print(
    non_zero_mismatches_df["h3_grid_distance"]
    .value_counts(dropna=False)
    .sort_index()
)

h3_grid_distance
1    26
Name: count, dtype: int64


In [20]:
from pathlib import Path

from initial_data_transformation import (
    InitialDataTransformation,
    TransformationValidationError,
)


repository_root = Path.cwd().parent

transformation = InitialDataTransformation(
    s3_client=s3_client,
    bucket=BUCKET,
    contract_path=repository_root
    / "config"
    / "data_transformation_contract.yml",
    log_path=repository_root
    / "logs"
    / "data_transformation.log",
)

try:
    transformation_result = transformation.run(
        validated_hexagons=result["features"]
    )
except TransformationValidationError as error:
    print(f"Transformation time: {error.time_seconds} seconds")
    print(f"H3 match rate: {error.match_rate}%")
    print(f"Failed join rate: {error.failed_join_rate}%")
    print("Transformation passed: False")
    print(f"Pipeline failed: {error}")
else:
    print(f"Transformation time: {transformation_result['time_seconds']} seconds")
    print(f"H3 match rate: {transformation_result['match_rate']}%")
    print(
        f"Failed join rate: "
        f"{transformation_result['failed_join_rate']}%"
    )
    print(f"Transformation passed: {transformation_result['passed']}")

    joined_requests_df = transformation_result["data"]
    display(joined_requests_df.head())

Transformation time: 24.0521 seconds
H3 match rate: 99.99692%
Failed join rate: 0.000411%
Transformation passed: True


,Unnamed: 0,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index
0,0,400583534,9.109492e+09,2020-10-07 06:55:18+02:00,2020-10-08 15:36:35+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area Central,District: Blaauwberg,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Wear and tear,MONTAGUE GARDENS,-33.872839,18.522488,88ad360225fffff
1,1,400555043,9.108995e+09,2020-07-09 16:08:13+02:00,2020-07-14 14:27:01+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,SOMERSET WEST,-34.078916,18.848940,88ad36d5e1fffff
2,2,400589145,9.109614e+09,2020-10-27 10:21:59+02:00,2020-10-28 17:48:15+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area East,District : Somerset West,TD Customer complaint groups,Manhole Cover/Gully Grid,Road (RCL),Vandalism,STRAND,-34.102242,18.821116,88ad36d437fffff
3,3,400538915,9.108601e+09,2020-03-19 06:36:06+02:00,2021-03-29 20:34:19+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Bellville,TD Customer complaint groups,Paint Markings Lines&Signs,Road Markings,Wear and tear,RAVENSMEAD,-33.920019,18.607209,88ad361133fffff
4,4,400568554,NaN,2020-08-25 09:48:42+02:00,2020-08-31 08:41:13+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,CLAREMONT,-33.987400,18.453760,88ad361709fffff


In [22]:
from time import perf_counter

start_time = perf_counter()

load_start = perf_counter()
service_requests = transformation._load_gzip_csv("sr.csv.gz")
expected_requests = transformation._load_gzip_csv("sr_hex.csv.gz")
load_seconds = perf_counter() - load_start

join_start = perf_counter()
joined_requests, valid_coordinates = transformation._spatial_join(
    service_requests=service_requests,
    validated_hexagons=result["features"],
    inputs={
        "latitude_column": "latitude",
        "longitude_column": "longitude",
        "polygon_index_column": "properties_index",
        "polygon_coordinates_column": "geometry_coordinates",
        "output_index_column": "h3_level8_index",
        "zero_index": "0",
    },
    spatial_join_contract={
        "coordinate_reference_system": "EPSG:4326",
        "predicate": "within",
        "latitude_range": [-90, 90],
        "longitude_range": [-180, 180],
        "require_single_polygon_match": True,
    },
)
join_seconds = perf_counter() - join_start

validation_start = perf_counter()
calculated_indexes = joined_requests["h3_level8_index"].fillna("0").astype(str)
expected_indexes = expected_requests["h3_level8_index"].fillna("0").astype(str)
match_rate = calculated_indexes.eq(expected_indexes).mean() * 100
validation_seconds = perf_counter() - validation_start

total_seconds = perf_counter() - start_time

print(f"S3 download and CSV parsing: {load_seconds:.2f} seconds")
print(f"Point creation and spatial join: {join_seconds:.2f} seconds")
print(f"Reference validation: {validation_seconds:.2f} seconds")
print(f"Total: {total_seconds:.2f} seconds")
print(f"Match rate: {match_rate:.6f}%")

S3 download and CSV parsing: 22.07 seconds
Point creation and spatial join: 2.29 seconds
Reference validation: 0.18 seconds
Total: 24.55 seconds
Match rate: 99.996920%


In [17]:
import importlib
import initial_data_transformation

importlib.reload(initial_data_transformation)

InitialDataTransformation = (
    initial_data_transformation.InitialDataTransformation
)
TransformationValidationError = (
    initial_data_transformation.TransformationValidationError
)

In [18]:
transformation = InitialDataTransformation(
    s3_client=s3_client,
    bucket=BUCKET,
    contract_path=repository_root
    / "config"
    / "data_transformation_contract.yml",
    log_path=repository_root
    / "logs"
    / "data_transformation.log",
)

In [19]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

import yaml


with (
    repository_root
    / "config"
    / "data_transformation_contract.yml"
).open() as file:
    contract = yaml.safe_load(file)

inputs = contract["inputs"]
spatial_join_contract = contract["spatial_join"]
workers = contract["performance"]["concurrent_download_workers"]

start_time = perf_counter()

load_start = perf_counter()

with ThreadPoolExecutor(max_workers=workers) as executor:
    service_requests_future = executor.submit(
        transformation._load_gzip_csv,
        inputs["service_requests_key"],
    )
    expected_requests_future = executor.submit(
        transformation._load_gzip_csv,
        inputs["validation_reference_key"],
        inputs["validation_reference_columns"],
    )

    service_requests = service_requests_future.result()
    expected_requests = expected_requests_future.result()

load_seconds = perf_counter() - load_start

join_start = perf_counter()

joined_requests, valid_coordinates = transformation._spatial_join(
    service_requests=service_requests,
    validated_hexagons=result["features"],
    inputs=inputs,
    spatial_join_contract=spatial_join_contract,
)

join_seconds = perf_counter() - join_start

validation_start = perf_counter()

calculated_indexes = (
    joined_requests[inputs["output_index_column"]]
    .fillna(inputs["zero_index"])
    .astype(str)
)

expected_indexes = (
    expected_requests[inputs["output_index_column"]]
    .fillna(inputs["zero_index"])
    .astype(str)
)

match_rate = calculated_indexes.eq(expected_indexes).mean() * 100

validation_seconds = perf_counter() - validation_start
total_seconds = perf_counter() - start_time

print(f"Concurrent S3 download and CSV parsing: {load_seconds:.2f} seconds")
print(f"Point creation and spatial join: {join_seconds:.2f} seconds")
print(f"Reference validation: {validation_seconds:.2f} seconds")
print(f"Total: {total_seconds:.2f} seconds")
print(f"Match rate: {match_rate:.6f}%")
print(f"Reference columns loaded: {expected_requests.columns.tolist()}")

Concurrent S3 download and CSV parsing: 19.09 seconds
Point creation and spatial join: 2.33 seconds
Reference validation: 0.17 seconds
Total: 21.59 seconds
Match rate: 99.996920%
Reference columns loaded: ['h3_level8_index']


# step 3

In [3]:
from urllib.parse import urlencode

import geopandas as gpd


OFFICIAL_SUBURBS_LAYER_URL = (
    "https://citymaps.capetown.gov.za/agsext/rest/services/"
    "Theme_Based/ODP_SPLIT_5/FeatureServer/3"
)

query_parameters = urlencode(
    {
        "where": "OFC_SBRB_NAME LIKE '%ATLANTIS%'",
        "outFields": "OBJECTID,OFC_SBRB_NAME",
        "returnGeometry": "true",
        "f": "geojson",
    }
)

atlantis_suburbs_gdf = gpd.read_file(
    f"{OFFICIAL_SUBURBS_LAYER_URL}/query?{query_parameters}"
)

if atlantis_suburbs_gdf.empty:
    raise ValueError(
        "The official suburb layer returned no suburbs matching 'ATLANTIS'."
    )

print(f"Atlantis-matching suburb polygons: {len(atlantis_suburbs_gdf)}")

display(
    atlantis_suburbs_gdf[
        ["OBJECTID", "OFC_SBRB_NAME", "geometry"]
    ]
)

print(atlantis_suburbs_gdf["OFC_SBRB_NAME"].tolist())

Atlantis-matching suburb polygons: 1


,OBJECTID,OFC_SBRB_NAME,geometry
0,580,ATLANTIS INDUSTRIAL,"POLYGON ((18.4823 -33.57684, 18.48208 -33.5767..."


['ATLANTIS INDUSTRIAL']


In [1]:
import json
import urllib.request
from pathlib import Path
import ssl
import boto3
import certifi
from data_extraction import DataExtraction, ExtractionValidationError


CREDENTIALS_URL = (
    "https://cct-ds-code-challenge-input-data.s3.af-south-1."
    "amazonaws.com/ds_code_challenge_creds.json"
)

BUCKET = "cct-ds-code-challenge-input-data"
REGION = "af-south-1"


def load_credentials():
    context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        CREDENTIALS_URL,
        context=context,
    ) as response:
        return json.load(response)["s3"]

credentials = load_credentials()

s3_client = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=credentials["access_key"],
    aws_secret_access_key=credentials["secret_key"],
)

In [7]:
from io import BytesIO
from urllib.parse import urlencode

import geopandas as gpd
import pandas as pd


OFFICIAL_SUBURBS_LAYER_URL = (
    "https://citymaps.capetown.gov.za/agsext/rest/services/"
    "Theme_Based/ODP_SPLIT_5/FeatureServer/3"
)

CANDIDATE_SUBURBS = [
    "ATLANTIS INDUSTRIAL",
    "SAXONSEA",
    "WESFLEUR",
    "WITSAND",
    "PELLA",
    "MAMRE",
]

WALKING_RADIUS_METRES = 80

suburb_names_sql = ", ".join(
    f"'{suburb_name}'"
    for suburb_name in CANDIDATE_SUBURBS
)

query_parameters = urlencode(
    {
        "where": f"OFC_SBRB_NAME IN ({suburb_names_sql})",
        "outFields": "OBJECTID,OFC_SBRB_NAME",
        "returnGeometry": "true",
        "f": "geojson",
    }
)

candidate_suburbs_gdf = gpd.read_file(
    f"{OFFICIAL_SUBURBS_LAYER_URL}/query?{query_parameters}"
)

if candidate_suburbs_gdf.empty:
    raise ValueError("No requested Atlantis-area suburbs were returned.")

# GeoJSON is longitude/latitude. Project to metres before centroid and distance work.
candidate_suburbs_m = candidate_suburbs_gdf.to_crs("EPSG:3857")
candidate_suburbs_m["centroid"] = candidate_suburbs_m.geometry.centroid

response = s3_client.get_object(
    Bucket=BUCKET,
    Key="sr_hex.csv.gz",
)

service_requests_df = pd.read_csv(
    BytesIO(response["Body"].read()),
    compression="gzip",
    usecols=["latitude", "longitude", "h3_level8_index"],
    low_memory=False,
)

latitude = pd.to_numeric(
    service_requests_df["latitude"],
    errors="coerce",
)

longitude = pd.to_numeric(
    service_requests_df["longitude"],
    errors="coerce",
)

valid_coordinates = (
    latitude.notna()
    & longitude.notna()
    & latitude.between(-90, 90)
    & longitude.between(-180, 180)
)

request_points = gpd.GeoDataFrame(
    service_requests_df.loc[valid_coordinates].copy(),
    geometry=gpd.points_from_xy(
        longitude.loc[valid_coordinates],
        latitude.loc[valid_coordinates],
    ),
    crs="EPSG:4326",
).to_crs("EPSG:3857")

candidate_results = []

for suburb in candidate_suburbs_m.itertuples():
    distances_metres = request_points.geometry.distance(suburb.centroid)
    request_count = distances_metres.le(WALKING_RADIUS_METRES).sum()

    candidate_results.append(
        {
            "suburb": suburb.OFC_SBRB_NAME,
            "suburb_object_id": suburb.OBJECTID,
            "requests_within_80m": int(request_count),
            "centroid_x_metres": round(suburb.centroid.x, 2),
            "centroid_y_metres": round(suburb.centroid.y, 2),
        }
    )

candidate_results_df = pd.DataFrame(
    candidate_results
).sort_values(
    "requests_within_80m",
    ascending=False,
)

print(f"Valid-coordinate requests assessed: {len(request_points):,}")
print(f"Walking-distance approximation: {WALKING_RADIUS_METRES} m")

display(candidate_results_df)

Valid-coordinate requests assessed: 729,270
Walking-distance approximation: 80 m


,suburb,suburb_object_id,requests_within_80m,centroid_x_metres,centroid_y_metres
1,WITSAND,413,48,2060040.78,-3973588.83
2,MAMRE,414,25,2056491.91,-3963730.99
3,PELLA,437,20,2061798.26,-3967012.09
0,SAXONSEA,150,3,2058089.02,-3968693.04
5,ATLANTIS INDUSTRIAL,580,3,2056675.07,-3974259.28
4,WESFLEUR,579,0,2056777.90,-3969838.63


In [2]:
import importlib

import further_data_transformation


importlib.reload(further_data_transformation)

FurtherDataTransformation = (
    further_data_transformation.FurtherDataTransformation
)
FurtherTransformationError = (
    further_data_transformation.FurtherTransformationError
)
repository_root = Path.cwd().parent
further_transformation = FurtherDataTransformation(
    s3_client=s3_client,
    bucket=BUCKET,
    contract_path=repository_root
    / "config"
    / "further_data_transformation_contract.yml",
    log_path=repository_root
    / "logs"
    / "further_data_transformation.log",
)

try:
    step_5_1_result = further_transformation.run()
except FurtherTransformationError as error:
    print(f"Step 5.1 failed: {error}")
else:
    witsand_subsample_df = step_5_1_result["data"]

    print(f"Selected suburb: {step_5_1_result['suburb_name']}")
    print(f"Source suburb object ID: {step_5_1_result['suburb_object_id']}")
    print(
        "Computed centroid in EPSG:32734: "
        f"({step_5_1_result['centroid_x']}, "
        f"{step_5_1_result['centroid_y']})"
    )
    print(f"Distance radius: {step_5_1_result['radius_metres']} m")
    print(f"Input service requests: {step_5_1_result['input_row_count']:,}")
    print(
        "Invalid coordinate records excluded: "
        f"{step_5_1_result['invalid_coordinate_count']:,}"
    )
    print(
        "Selected service requests: "
        f"{step_5_1_result['selected_row_count']:,}"
    )
    print(f"Step 5.1 time: {step_5_1_result['time_seconds']} seconds")

    display(witsand_subsample_df.head())

Selected suburb: WITSAND
Source suburb object ID: 413
Computed centroid in EPSG:32734: (268518.019, 6280756.384)
Distance radius: 80 m
Input service requests: 941,634
Invalid coordinate records excluded: 212,364
Selected service requests: 76
Step 5.1 time: 17.9637 seconds


,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index
49371,1015479905,NaN,2020-01-07 11:50:41+02:00,2020-05-05 13:04:41+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Sewerline Defective,NaN,NaN,WITSAND,-33.587425,18.505019,88ad36701dfffff
49450,1015479992,NaN,2020-01-07 11:55:58+02:00,2020-01-21 09:11:34+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Manhole Blocked,NaN,NaN,WITSAND,-33.587644,18.505621,88ad36701dfffff
124007,1015564614,NaN,2020-01-31 14:35:02+02:00,2020-02-12 12:40:30+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,WATER - INFORMAL SETTLEMENTS,Tap Missing,NaN,NaN,WITSAND,-33.587283,18.505515,88ad36701dfffff
157866,1015603122,NaN,2020-02-12 12:19:43+02:00,2020-02-21 14:30:50+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Pan Connector Defective,NaN,NaN,WITSAND,-33.587372,18.505385,88ad36701dfffff
166111,1015612622,9.108423e+09,2020-02-14 20:13:11+02:00,2020-02-17 10:07:11+02:00,WATER AND SANITATION,Distribution Services,Reticulation,NaN,SEWER,Sewer: Blocked/Overflow,NaN,NaN,WITSAND,-33.587255,18.505137,88ad36701dfffff


In [24]:
from io import BytesIO
import ssl
from urllib.request import Request, urlopen

import certifi
import pandas as pd


WIND_DATA_URL = (
    "https://www.arcgis.com/sharing/rest/content/items/"
    "31ef242a23484e79bbb19d6b29203179/data"
)

ssl_context = ssl.create_default_context(
    cafile=certifi.where()
)

request = Request(
    WIND_DATA_URL,
    headers={"User-Agent": "Mozilla/5.0"},
)

with urlopen(
    request,
    timeout=30,
    context=ssl_context,
) as response:
    wind_workbook_bytes = response.read()

if not wind_workbook_bytes.startswith(b"PK"):
    raise ValueError(
        "Wind download is not an XLSX workbook: expected ZIP-format bytes."
    )

wind_workbook = pd.ExcelFile(
    BytesIO(wind_workbook_bytes),
    engine="openpyxl",
)

print(f"Worksheet names: {wind_workbook.sheet_names}")

for sheet_name in wind_workbook.sheet_names:
    wind_preview_df = pd.read_excel(
        wind_workbook,
        sheet_name=sheet_name,
        engine="openpyxl",
        nrows=10,
    )

    print(f"\nWorksheet: {sheet_name}")
    print(f"Columns: {wind_preview_df.columns.tolist()}")
    display(wind_preview_df)

Worksheet names: ['Sheet1']

Worksheet: Sheet1
Columns: ['MultiStation:  Periodically: 01/01/2020 00:00-31/12/2020 23:59  Type: AVG 1 Hr.', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']


,MultiStation: Periodically: 01/01/2020 00:00-31/12/2020 23:59 Type: AVG 1 Hr.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date & Time,Atlantis AQM Site,Atlantis AQM Site,Bellville South AQM Site,Bellville South AQM Site,Bothasig AQM Site,Bothasig AQM Site,Goodwood AQM Station,Goodwood AQM Station,Khayelitsha AQM Site,Khayelitsha AQM Site,Somerset West AQM Site,Somerset West AQM Site,Tableview AQM Site,Tableview AQM Site
2,NaN,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V,Wind Dir V,Wind Speed V
3,NaN,Deg,m/s,Deg,m/s,Deg,m/s,Deg,m/s,Deg,m/s,Deg,m/s,Deg,m/s
4,01/01/2020 00:00,173,4.1,191,2.5,163.7,5.3,247.8,19.2,34.2,1.3,135,3.8,179.8,5.2
5,01/01/2020 01:00,177.7,4,209.7,1.6,159,5.4,247,17.9,34.9,1.1,132.7,2.1,177.9,5.2
6,01/01/2020 02:00,180.7,2.8,202.5,1.4,148.8,5.5,246.4,17.1,35.5,1.1,128.5,2.4,167.8,4
7,01/01/2020 03:00,183.7,2.3,224.7,1.2,153,4.7,245.1,15.7,35.5,1,357.6,1.1,177.3,4.4
8,01/01/2020 04:00,170.7,2.4,244.3,1.3,153.4,4.1,249.9,15.8,35.1,0.8,319.5,1.4,178.7,3.8
9,01/01/2020 05:00,195.5,2.2,245,1.6,151.2,4,249.7,14.1,36.6,0.6,316.1,1.6,175.8,4


In [4]:
from io import BytesIO
import ssl
import time
from urllib.error import URLError
from urllib.request import Request, urlopen

import certifi
import pandas as pd


WIND_DATA_URL = (
    "https://www.arcgis.com/sharing/rest/content/items/"
    "31ef242a23484e79bbb19d6b29203179/data"
)
MAX_ATTEMPTS = 3
RETRY_DELAY_SECONDS = 2

ATLANTIS_DIRECTION_COLUMN_INDEX = 1
ATLANTIS_SPEED_COLUMN_INDEX = 2
NOTIFICATION_TIME_COLUMN = "creation_timestamp"  # confirm against sr_hex.csv.gz


def download_wind_workbook(url, max_attempts, retry_delay_seconds):
    ssl_context = ssl.create_default_context(cafile=certifi.where())
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})

    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            with urlopen(request, timeout=30, context=ssl_context) as response:
                workbook_bytes = response.read()

            if not workbook_bytes.startswith(b"PK"):
                raise ValueError("Downloaded content is not a valid XLSX workbook.")

            return workbook_bytes

        except (URLError, ValueError) as error:
            last_error = error
            if attempt < max_attempts:
                time.sleep(retry_delay_seconds * attempt)

    raise RuntimeError(
        f"Wind data source unavailable after {max_attempts} attempts."
    ) from last_error


# Retries with backoff, not a synthetic fallback, so an unreachable source
# fails loudly instead of silently corrupting the augmented dataset.
wind_workbook_bytes = download_wind_workbook(
    WIND_DATA_URL, MAX_ATTEMPTS, RETRY_DELAY_SECONDS
)

# Rows 0-3 are the multi-level station/metric/unit header; data starts row 4.
raw_wind_df = pd.read_excel(
    BytesIO(wind_workbook_bytes),
    engine="openpyxl",
    header=None,
    skiprows=4,
)

atlantis_wind_df = pd.DataFrame(
    {
        "wind_timestamp": pd.to_datetime(
            raw_wind_df[0], format="%d/%m/%Y %H:%M", errors="coerce"
        ),
        "wind_direction_deg": pd.to_numeric(
            raw_wind_df[ATLANTIS_DIRECTION_COLUMN_INDEX], errors="coerce"
        ),
        "wind_speed_ms": pd.to_numeric(
            raw_wind_df[ATLANTIS_SPEED_COLUMN_INDEX], errors="coerce"
        ),
    }
).dropna(subset=["wind_timestamp"]).sort_values("wind_timestamp")

witsand_subsample_df[NOTIFICATION_TIME_COLUMN] = pd.to_datetime(
    witsand_subsample_df[NOTIFICATION_TIME_COLUMN]
).dt.tz_localize(None)
witsand_subsample_df = witsand_subsample_df.sort_values(NOTIFICATION_TIME_COLUMN)

augmented_df = pd.merge_asof(
    witsand_subsample_df,
    atlantis_wind_df,
    left_on=NOTIFICATION_TIME_COLUMN,
    right_on="wind_timestamp",
    direction="nearest",
)

print(f"Atlantis wind observations: {len(atlantis_wind_df):,}")
print(f"Augmented requests: {len(augmented_df):,}")
display(augmented_df.head())

Atlantis wind observations: 8,784
Augmented requests: 76


,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index,wind_timestamp,wind_direction_deg,wind_speed_ms
0,1015479905,NaN,2020-01-07 11:50:41,2020-05-05 13:04:41+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Sewerline Defective,NaN,NaN,WITSAND,-33.587425,18.505019,88ad36701dfffff,2020-01-07 12:00:00,49.1,3.9
1,1015479992,NaN,2020-01-07 11:55:58,2020-01-21 09:11:34+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Manhole Blocked,NaN,NaN,WITSAND,-33.587644,18.505621,88ad36701dfffff,2020-01-07 12:00:00,49.1,3.9
2,1015564614,NaN,2020-01-31 14:35:02,2020-02-12 12:40:30+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,WATER - INFORMAL SETTLEMENTS,Tap Missing,NaN,NaN,WITSAND,-33.587283,18.505515,88ad36701dfffff,2020-01-31 15:00:00,217.2,7.0
3,1015603122,NaN,2020-02-12 12:19:43,2020-02-21 14:30:50+02:00,WATER AND SANITATION,Distribution Services,Informal Settlements Basic Services,Informal Settlements:Operating and Maintenance,SEWER - INFORMAL SETTLEMENTS,Pan Connector Defective,NaN,NaN,WITSAND,-33.587372,18.505385,88ad36701dfffff,2020-02-12 12:00:00,228.8,3.2
4,1015612622,9.108423e+09,2020-02-14 20:13:11,2020-02-17 10:07:11+02:00,WATER AND SANITATION,Distribution Services,Reticulation,NaN,SEWER,Sewer: Blocked/Overflow,NaN,NaN,WITSAND,-33.587255,18.505137,88ad36701dfffff,2020-02-14 20:00:00,174.6,4.6
